# M1.0c Primer: Gradient, Divergence, Curl and Laplacian

Book pointer: chapter 1, section 1.4 (differentiation).

Third of four primer notebooks. This is the heart of vector calculus: four operators that describe how a field changes from point to point. Each one is introduced as a measurement you could make with a tiny instrument, derived from that measurement, and only then written as a formula.

## 1. Warm-up

1. What is a partial derivative? How do you compute $\partial f/\partial y$ for $f = x^2 y + \sin x$?
2. Which way does $\nabla f$ point, and what is its length?
3. In words: divergence measures what? Curl measures what? Which one is a scalar?
4. Can a field have zero divergence but non-zero curl? Give an example. The other way round?
5. What does it mean for a function to satisfy $\nabla^2 f = 0$, in terms of its neighbours?

## 2. The idea in one paragraph

A derivative is a ratio of a small change in output to a small change in input. With several inputs there are several such ratios, the partial derivatives, and on a grid each one is a difference between neighbouring values divided by the spacing. The four operators of this notebook are combinations of partial derivatives chosen because each answers a physical question: the gradient asks which way is uphill, the divergence asks whether a tiny box gains or loses stuff, the curl asks whether a tiny paddlewheel spins, and the Laplacian asks whether a point sits above or below the average of its neighbours. Each of these questions can be asked with a measurement, and the formula is what the measurement converges to as the instrument shrinks.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=5, suppress=True)

### 3.1 Partial derivatives and finite differences

For $f(x, y, z)$, the partial derivative $\partial f/\partial x$ is the ordinary derivative with respect to $x$ while $y$ and $z$ are held fixed. Rules carry over unchanged: differentiate term by term, treat the other variables as constants.

**Worked example 1.** $f = x^2y + \sin x - y^3$. $\partial f/\partial x = 2xy + \cos x$ (the $y^3$ term has no $x$ and drops out). $\partial f/\partial y = x^2 - 3y^2$ (the $\sin x$ drops out). At $(1, 2)$: $\partial f/\partial x = 4 + 0.5403 = 4.5403$ and $\partial f/\partial y = 1 - 12 = -11$.

On a computer we only have $f$ at grid points, so a derivative becomes a difference quotient. The forward difference $(f(x + h) - f(x))/h$ has an error proportional to $h$. The central difference $(f(x + h) - f(x - h))/2h$ has an error proportional to $h^2$, because the first-order errors on the two sides cancel. `np.gradient` uses central differences inside an array and one-sided ones at the edges. The table shows the two error rates.

In [ ]:
f = lambda x, y: x**2 * y + np.sin(x) - y**3
fx_exact = lambda x, y: 2 * x * y + np.cos(x)
x0, y0 = 1.0, 2.0
print(f"exact df/dx at (1, 2) = {fx_exact(x0, y0):.6f}      exact df/dy = {x0**2 - 3 * y0**2}")
print("      h      forward error    central error")
for h in (0.1, 0.01, 0.001, 0.0001):
    fwd = (f(x0 + h, y0) - f(x0, y0)) / h
    cen = (f(x0 + h, y0) - f(x0 - h, y0)) / (2 * h)
    print(f"{h:8.4f}   {abs(fwd - fx_exact(x0, y0)):12.2e}   {abs(cen - fx_exact(x0, y0)):12.2e}")
print("forward: error falls 10x per 10x smaller h (first order). central: 100x (second order).")

### 3.2 The gradient

Put the partial derivatives of a scalar field together as a vector:
$$\nabla f = \frac{\partial f}{\partial x}\hat{\mathbf{x}} + \frac{\partial f}{\partial y}\hat{\mathbf{y}} + \frac{\partial f}{\partial z}\hat{\mathbf{z}}.$$

Why this vector? Move from $\mathbf{r}$ by a small step $d\mathbf{l} = (dx, dy, dz)$. The change in $f$ is, to first order,
$$df = \frac{\partial f}{\partial x}dx + \frac{\partial f}{\partial y}dy + \frac{\partial f}{\partial z}dz = \nabla f\cdot d\mathbf{l}.$$
That dot product tells the whole story. For a step of fixed length, $df$ is largest when $d\mathbf{l}$ is parallel to $\nabla f$: the gradient points in the direction of steepest increase. Its length is the slope in that direction. Stepping perpendicular to $\nabla f$ gives $df = 0$, which means you are walking along a contour: the gradient is perpendicular to the contour lines. And the slope in an arbitrary unit direction $\hat{\mathbf{u}}$, the directional derivative, is $\nabla f\cdot\hat{\mathbf{u}}$.

In chapter 3, $\mathbf{E} = -\nabla V$: the electric field points downhill in potential, perpendicular to the equipotential surfaces, with a strength equal to the slope.

**Worked example 2.** Bowl $f = \tfrac12x^2 + y^2 - \tfrac12xy$: $\nabla f = (x - \tfrac12y,\; 2y - \tfrac12x)$. At $(1, 0.5)$: $(0.75, 0.5)$, length $0.901$. Slope towards $(1, 1)/\sqrt2$: $(0.75 + 0.5)/\sqrt2 = 0.884$. Slope towards $(1, 0)$: $0.75$. Slope along the contour direction, perpendicular to the gradient, $(-0.5, 0.75)/0.901$: $(-0.375 + 0.375)/0.901 = 0$.

**Worked example 3.** $f = 1/r$ with $r = \sqrt{x^2 + y^2 + z^2}$. By the chain rule $\partial r/\partial x = x/r$, so $\partial(1/r)/\partial x = -x/r^3$, and $\nabla(1/r) = -(x, y, z)/r^3 = -\hat{\mathbf{r}}/r^2$. The potential of a point charge is proportional to $1/r$ and its field to $-\nabla(1/r) = \hat{\mathbf{r}}/r^2$: Coulomb's law falls out of the gradient.

In [ ]:
x = np.linspace(-2, 2, 81); y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y, indexing="ij"); h = x[1] - x[0]
bowl = X**2 / 2 + Y**2 - X * Y / 2
gx, gy = np.gradient(bowl, h, h)

fig, ax = plt.subplots(figsize=(6.5, 6))
cs = ax.contour(X, Y, bowl, levels=15, cmap="viridis"); ax.clabel(cs, fontsize=7)
st = 6
ax.quiver(X[::st, ::st], Y[::st, ::st], gx[::st, ::st], gy[::st, ::st], color="k")
ax.set_aspect("equal"); ax.set_title("gradient arrows: perpendicular to contours, pointing uphill, longer where steeper"); plt.show()

p = np.array([1.0, 0.5]); grad_p = np.array([p[0] - p[1] / 2, 2 * p[1] - p[0] / 2])
bowl_f = lambda q: q[0]**2 / 2 + q[1]**2 - q[0] * q[1] / 2
for name, u in (("(1,1)/sqrt2", np.array([1, 1]) / np.sqrt(2)), ("(1,0)", np.array([1.0, 0])), ("along contour", np.array([-0.5, 0.75]) / 0.901)):
    e = 1e-5
    print(f"slope in direction {name:14s}: grad . u = {grad_p @ u:+.4f}   direct finite difference = {(bowl_f(p + e * u) - bowl_f(p - e * u)) / (2 * e):+.4f}")

A gradient has a use beyond physics that makes its meaning stick: walking downhill. Starting anywhere, take repeated steps of $-\alpha\nabla f$ and you descend towards a minimum. The path crosses every contour at a right angle. (This is gradient descent, the workhorse of optimisation.)

In [ ]:
grad_bowl = lambda q: np.array([q[0] - q[1] / 2, 2 * q[1] - q[0] / 2])
path = [np.array([1.8, 1.5])]
for _ in range(25):
    path.append(path[-1] - 0.25 * grad_bowl(path[-1]))
path = np.array(path)
fig, ax = plt.subplots(figsize=(6, 6))
ax.contour(X, Y, bowl, levels=20, cmap="viridis")
ax.plot(path[:, 0], path[:, 1], "o-", color="C3", ms=4, label="steps of -0.25 grad f")
ax.set_aspect("equal"); ax.legend(); ax.set_title("walking downhill with the gradient"); plt.show()
print("final point", path[-1], "  minimum is at (0, 0)")

### 3.3 The divergence

Imagine a vector field as the velocity of a fluid. Draw a tiny box of sides $\Delta x, \Delta y, \Delta z$ around a point. The flux out through the right face (normal $+\hat{\mathbf{x}}$) is $F_x(x + \Delta x/2)\,\Delta y\Delta z$, and through the left face (normal $-\hat{\mathbf{x}}$) it is $-F_x(x - \Delta x/2)\,\Delta y\Delta z$. Their sum is
$$\big[F_x(x + \tfrac12\Delta x) - F_x(x - \tfrac12\Delta x)\big]\Delta y\Delta z \approx \frac{\partial F_x}{\partial x}\Delta x\Delta y\Delta z.$$
The $y$ and $z$ pairs of faces give the same with $F_y$ and $F_z$. Divide the total outward flux by the box volume and let the box shrink:
$$\nabla\cdot\mathbf{F} = \lim_{\Delta v\to0}\frac{\text{flux out of the box}}{\Delta v} = \frac{\partial F_x}{\partial x} + \frac{\partial F_y}{\partial y} + \frac{\partial F_z}{\partial z}.$$

So divergence is outward flux per unit volume: positive where the field is being created (a source, fluid appearing), negative where it is being destroyed (a sink), zero where whatever enters also leaves. In chapter 4, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ says that electric charge is exactly where electric field lines start and end.

**Worked example 4.** The six reference fields from notebook 00a, in 2D ($\nabla\cdot\mathbf{F} = \partial F_x/\partial x + \partial F_y/\partial y$):

| field | $F_x$ | $F_y$ | $\partial F_x/\partial x$ | $\partial F_y/\partial y$ | div |
|---|---|---|---|---|---|
| uniform | 1 | 0 | 0 | 0 | 0 |
| radial | $x$ | $y$ | 1 | 1 | 2 |
| rotation | $-y$ | $x$ | 0 | 0 | 0 |
| saddle | $x$ | $-y$ | 1 | $-1$ | 0 |
| shear | $y$ | 0 | 0 | 0 | 0 |
| sink+swirl | $-x-y$ | $x-y$ | $-1$ | $-1$ | $-2$ |

**Worked example 5.** $\mathbf{F} = (x^2, xy, z)$ in 3D: $\nabla\cdot\mathbf{F} = 2x + x + 1 = 3x + 1$. It depends on position: a source for $x > -1/3$, a sink for $x < -1/3$.

The code measures the divergence the way it was defined, as flux out of a shrinking square divided by its area, and compares with the table.

In [ ]:
def flux_out_of_square(Fx, Fy, cx, cy, half, n=200):
    """Outward flux of (Fx, Fy) through the square of half-width half centred at (cx, cy)."""
    s = cx + np.linspace(-half, half, n); t = cy + np.linspace(-half, half, n)
    ds = s[1] - s[0]
    right  = np.trapezoid(Fx(np.full_like(t, cx + half), t), dx=ds)
    left   = np.trapezoid(-Fx(np.full_like(t, cx - half), t), dx=ds)
    top    = np.trapezoid(Fy(s, np.full_like(s, cy + half)), dx=ds)
    bottom = np.trapezoid(-Fy(s, np.full_like(s, cy - half)), dx=ds)
    return right + left + top + bottom

fields_2d = {
    "uniform":    (lambda x, y: np.ones_like(x), lambda x, y: np.zeros_like(x)),
    "radial":     (lambda x, y: x, lambda x, y: y),
    "rotation":   (lambda x, y: -y, lambda x, y: x),
    "saddle":     (lambda x, y: x, lambda x, y: -y),
    "shear":      (lambda x, y: y, lambda x, y: np.zeros_like(x)),
    "sink+swirl": (lambda x, y: -x - y, lambda x, y: x - y),
}
print("flux / area of shrinking squares centred at (0.7, -0.4):")
for name, (Fx, Fy) in fields_2d.items():
    row = [flux_out_of_square(Fx, Fy, 0.7, -0.4, half) / (2 * half) ** 2 for half in (1.0, 0.1, 0.01)]
    print(f"  {name:11s} {np.round(row, 4)}")

Fx5, Fy5 = (lambda x, y: x**2), (lambda x, y: x * y)      # the 2D part of worked example 5: div = 2x + x = 3x
for cx in (-1.0, 0.0, 0.5):
    print(f"  (x^2, xy) at x = {cx:+.1f}: measured {flux_out_of_square(Fx5, Fy5, cx, 0.3, 0.001) / 0.002**2:.4f}   formula 3x = {3 * cx:+.1f}")

### 3.4 The curl

Now the other instrument: a tiny paddlewheel, or equivalently a tiny closed loop. Take a square loop of sides $\Delta x, \Delta y$ in the $xy$ plane, traversed counter-clockwise, and add up $\mathbf{F}\cdot d\mathbf{l}$ around it. Along the bottom edge (moving in $+x$, at height $y - \Delta y/2$) the contribution is $F_x(y - \Delta y/2)\Delta x$; along the top edge (moving in $-x$) it is $-F_x(y + \Delta y/2)\Delta x$. Together: $-\frac{\partial F_x}{\partial y}\Delta x\Delta y$. The right and left edges give $+\frac{\partial F_y}{\partial x}\Delta x\Delta y$. Divide by the area and shrink:
$$(\nabla\times\mathbf{F})_z = \lim_{\Delta S\to0}\frac{\text{circulation around the loop}}{\Delta S} = \frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}.$$

Loops in the other two planes give the $x$ and $y$ components, with the indices cycling ($x \to yz$, $y \to zx$):
$$\nabla\times\mathbf{F} = \left(\frac{\partial F_z}{\partial y} - \frac{\partial F_y}{\partial z}\right)\hat{\mathbf{x}} + \left(\frac{\partial F_x}{\partial z} - \frac{\partial F_z}{\partial x}\right)\hat{\mathbf{y}} + \left(\frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}\right)\hat{\mathbf{z}}.$$
Same pattern as the cross product in 00a with $\nabla$ as the first vector, which is why it is written $\nabla\times\mathbf{F}$.

So curl is circulation per unit area, a vector whose direction is the axis a paddlewheel would spin about (right-hand rule) and whose length is twice the spin rate. In chapter 14, $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$ says a changing magnetic field makes the electric field circulate. In chapter 12, $\nabla\times\mathbf{B} = \mu_0\mathbf{J}$ says current makes the magnetic field circulate around it.

**Worked example 6.** The reference fields, $z$ component $\partial F_y/\partial x - \partial F_x/\partial y$:

| field | $F_x$ | $F_y$ | $\partial F_y/\partial x$ | $\partial F_x/\partial y$ | curl$_z$ |
|---|---|---|---|---|---|
| uniform | 1 | 0 | 0 | 0 | 0 |
| radial | $x$ | $y$ | 0 | 0 | 0 |
| rotation | $-y$ | $x$ | 1 | $-1$ | 2 |
| saddle | $x$ | $-y$ | 0 | 0 | 0 |
| shear | $y$ | 0 | 0 | 1 | $-1$ |
| sink+swirl | $-x-y$ | $x-y$ | 1 | $-1$ | 2 |

The shear field, all arrows parallel, has curl $-1$: a paddlewheel in it spins clockwise because the flow is faster on its upper side. The sink-with-swirl field has both divergence $-2$ and curl $2$. Divergence and curl are independent: any combination is possible.

**Worked example 7.** $\mathbf{F} = (yz, zx, xy)$ in 3D. $x$: $\partial F_z/\partial y - \partial F_y/\partial z = x - x = 0$. $y$: $\partial F_x/\partial z - \partial F_z/\partial x = y - y = 0$. $z$: $\partial F_y/\partial x - \partial F_x/\partial y = z - z = 0$. Curl zero everywhere. Indeed $\mathbf{F} = \nabla(xyz)$, and the curl of a gradient is always zero (section 3.6).

In [ ]:
def circulation_around_square(Fx, Fy, cx, cy, half, n=200):
    """Counter-clockwise line integral of (Fx, Fy) around the square of half-width half centred at (cx, cy)."""
    s = np.linspace(-half, half, n); ds = s[1] - s[0]
    bottom = np.trapezoid(Fx(cx + s, np.full_like(s, cy - half)), dx=ds)      # moving +x
    right  = np.trapezoid(Fy(np.full_like(s, cx + half), cy + s), dx=ds)      # moving +y
    top    = np.trapezoid(-Fx(cx + s, np.full_like(s, cy + half)), dx=ds)     # moving -x
    left   = np.trapezoid(-Fy(np.full_like(s, cx - half), cy + s), dx=ds)     # moving -y
    return bottom + right + top + left

print("circulation / area of shrinking loops centred at (0.7, -0.4):")
for name, (Fx, Fy) in fields_2d.items():
    row = [circulation_around_square(Fx, Fy, 0.7, -0.4, half) / (2 * half) ** 2 for half in (1.0, 0.1, 0.01)]
    print(f"  {name:11s} {np.round(row, 4)}")

# The paddlewheel picture for the shear field: the flow is faster above the wheel than below it.
fig, ax = plt.subplots(figsize=(6, 4))
yy = np.linspace(-1, 1, 9)
ax.quiver(np.zeros_like(yy) - 1.5, yy, yy, np.zeros_like(yy), scale=6, color="grey")
ax.add_patch(plt.Circle((0, 0.4), 0.3, fill=False, lw=2, color="C3"))
ax.annotate("", xy=(0.3, 0.4), xytext=(0.28, 0.55), arrowprops=dict(arrowstyle="->", color="C3", lw=2))
ax.text(0.4, 0.7, "faster flow on top pushes\nthe wheel clockwise: curl_z = -1", color="C3")
ax.set_xlim(-2, 2.5); ax.set_ylim(-1.2, 1.2); ax.set_aspect("equal"); ax.set_title("shear field F = (y, 0)"); ax.axis("off"); plt.show()

### 3.5 The Laplacian

The Laplacian is the divergence of the gradient, $\nabla^2 f = \nabla\cdot(\nabla f) = \partial^2 f/\partial x^2 + \partial^2 f/\partial y^2 + \partial^2 f/\partial z^2$. It appears when the two first-order operators are combined, which happens in chapter 6: $\mathbf{E} = -\nabla V$ and $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ give $\nabla^2 V = -\rho/\epsilon_0$, Poisson's equation.

Its meaning on a grid is simple. The second difference $(f_{i+1} - 2f_i + f_{i-1})/h^2$ approximates $\partial^2 f/\partial x^2$. Adding the same for $y$ gives
$$\nabla^2 f \approx \frac{f_{i+1,j} + f_{i-1,j} + f_{i,j+1} + f_{i,j-1} - 4f_{i,j}}{h^2} = \frac{4}{h^2}\big(\text{average of the four neighbours} - f_{i,j}\big).$$
So the Laplacian measures how far a point sits below the average of its surroundings. A function with $\nabla^2 f = 0$ (harmonic) equals the average of its neighbours everywhere; it has no bumps or hollows, only saddles. That is why the electrostatic potential in empty space has no local maxima or minima, and it is why relaxation methods in module 2 work: keep replacing each value by its neighbour average until nothing changes.

**Worked example 8.** Bowl $f = \tfrac12x^2 + y^2 - \tfrac12xy$: $f_{xx} = 1$, $f_{yy} = 2$, $\nabla^2 f = 3 > 0$. The bowl sits below its neighbours everywhere (it curves up). Saddle $f = x^2 - y^2$: $2 - 2 = 0$, harmonic. Hill $f = -x^2 - y^2$: $-4$, above its neighbours.

**Worked example 9.** $f = 1/r$ in 3D away from the origin. From worked example 3, $\partial f/\partial x = -x/r^3$. Differentiate again: $\partial^2 f/\partial x^2 = -1/r^3 + 3x^2/r^5$. Sum over $x, y, z$: $-3/r^3 + 3(x^2 + y^2 + z^2)/r^5 = -3/r^3 + 3/r^3 = 0$. The Coulomb potential is harmonic everywhere except at the charge. (In 2D the harmonic radial function is $\ln r$, not $1/r$.)

In [ ]:
def laplacian_grid(F, h):
    return (F[2:, 1:-1] + F[:-2, 1:-1] + F[1:-1, 2:] + F[1:-1, :-2] - 4 * F[1:-1, 1:-1]) / h**2

for name, F, exact in (("bowl", bowl, 3.0), ("saddle x^2 - y^2", X**2 - Y**2, 0.0), ("hill -x^2 - y^2", -X**2 - Y**2, -4.0)):
    L = laplacian_grid(F, h)
    print(f"{name:18s} Laplacian on the grid: mean {L.mean():+.4f}, spread {L.std():.1e}   exact {exact:+.1f}")

# Mean-value property: average of a harmonic function over a circle equals its centre value.
t = np.linspace(0, 2 * np.pi, 2001)[:-1]
for name, func, centre in (("saddle (harmonic)", lambda x, y: x**2 - y**2, (0.6, 0.3)), ("bowl (not harmonic)", lambda x, y: x**2 / 2 + y**2 - x * y / 2, (0.6, 0.3))):
    cx, cy = centre
    avg = np.mean(func(cx + 0.5 * np.cos(t), cy + 0.5 * np.sin(t)))
    print(f"{name:20s} average on a circle of radius 0.5: {avg:.4f}   value at the centre: {func(cx, cy):.4f}")

# Laplacian of 1/r in 3D, away from the origin.
s = np.linspace(1, 2, 41); hs = s[1] - s[0]
X3, Y3, Z3 = np.meshgrid(s, s, s, indexing="ij")
inv_r = 1 / np.sqrt(X3**2 + Y3**2 + Z3**2)
lap3 = sum(np.gradient(np.gradient(inv_r, hs, axis=a), hs, axis=a) for a in range(3))
print(f"Laplacian of 1/r on a 3D grid away from the origin: max |value| = {np.abs(lap3[2:-2, 2:-2, 2:-2]).max():.1e}  (typical |1/r| there ~ {inv_r.mean():.2f})")

### 3.6 Two identities that are always true

$$\nabla\times(\nabla f) = 0 \qquad\text{and}\qquad \nabla\cdot(\nabla\times\mathbf{F}) = 0.$$

Both follow from the fact that mixed partial derivatives commute, $\partial^2 f/\partial x\partial y = \partial^2 f/\partial y\partial x$. For the first: the $z$ component of $\nabla\times(\nabla f)$ is $\partial_x(\partial_y f) - \partial_y(\partial_x f) = 0$. For the second, every term appears twice with opposite signs.

They matter because they run backwards. A field with zero curl is the gradient of some scalar (that is how the electric potential $V$ exists in chapter 3). A field with zero divergence is the curl of some vector (that is how the magnetic vector potential $\mathbf{A}$ exists in chapter 11). The code verifies both identities symbolically with SymPy, which does the algebra exactly, and then numerically on a grid.

**Worked example 10.** $f = x^2y + \sin z$. $\nabla f = (2xy, x^2, \cos z)$. Curl: $x$: $\partial_y\cos z - \partial_z x^2 = 0$; $y$: $\partial_z(2xy) - \partial_x\cos z = 0$; $z$: $\partial_x x^2 - \partial_y(2xy) = 2x - 2x = 0$.

In [ ]:
import sympy as sp
xs, ys, zs = sp.symbols("x y z")
f_sym = xs**2 * ys + sp.sin(zs)
grad = [sp.diff(f_sym, v) for v in (xs, ys, zs)]
def curl(F):
    return [sp.diff(F[2], ys) - sp.diff(F[1], zs), sp.diff(F[0], zs) - sp.diff(F[2], xs), sp.diff(F[1], xs) - sp.diff(F[0], ys)]
def div(F):
    return sp.diff(F[0], xs) + sp.diff(F[1], ys) + sp.diff(F[2], zs)
print("grad f =", grad, "    curl(grad f) =", [sp.simplify(c) for c in curl(grad)])
G = [ys * zs**2, sp.exp(xs) * zs, xs * ys**3]
print("curl G =", curl(G), "    div(curl G) =", sp.simplify(div(curl(G))))

# The same two identities numerically, on the 3D grid from the previous cell.
F3 = X3**2 * Y3 + np.sin(Z3)
g = np.stack(np.gradient(F3, hs, hs, hs), axis=-1)
curl_g = np.stack([np.gradient(g[..., 2], hs, axis=1) - np.gradient(g[..., 1], hs, axis=2),
                   np.gradient(g[..., 0], hs, axis=2) - np.gradient(g[..., 2], hs, axis=0),
                   np.gradient(g[..., 1], hs, axis=0) - np.gradient(g[..., 0], hs, axis=1)], axis=-1)
print(f"numerical |curl(grad f)| max = {np.abs(curl_g[2:-2, 2:-2, 2:-2]).max():.1e}  (typical |grad f| ~ {np.linalg.norm(g, axis=-1).mean():.2f})")

### Exercise: measure a curl yourself

Replace the body of `curl_by_shrinking_loop` so that it estimates $(\nabla\times\mathbf{F})_z$ at a point from the circulation around a small square centred there divided by its area. The check uses $\mathbf{F} = (-y^2, x)$, whose curl is $1 + 2y$.

In [ ]:
def curl_by_shrinking_loop(Fx, Fy, cx, cy, half=1e-3):
    """Estimate the z component of curl F at (cx, cy) from a square loop of half-width half."""
    return circulation_around_square(Fx, Fy, cx, cy, half) / (2 * half) ** 2        # replace with your own version


Fx_t, Fy_t = (lambda x, y: -y**2), (lambda x, y: x)
for cy in (-1.0, 0.0, 0.5, 1.5):
    est = curl_by_shrinking_loop(Fx_t, Fy_t, 0.3, cy)
    print(f"at y = {cy:+.1f}: measured {est:.4f}   exact 1 + 2y = {1 + 2 * cy:.4f}")
    assert abs(est - (1 + 2 * cy)) < 1e-3
print("curl exercise passed")

## 4. Verify

Product rules. Vector calculus has a handful of them, listed in `study/vector_calculus_ref.md`. Two are used constantly in the course:
$$\nabla\cdot(f\mathbf{A}) = f\,\nabla\cdot\mathbf{A} + \mathbf{A}\cdot\nabla f, \qquad \nabla\times(f\mathbf{A}) = f\,\nabla\times\mathbf{A} + \nabla f\times\mathbf{A}.$$
Each is the ordinary product rule applied term by term. The check is symbolic, with SymPy, on a non-trivial $f$ and $\mathbf{A}$, so that the identities hold exactly rather than approximately.

In [ ]:
f_sym = xs * ys**2 + sp.cos(zs)
A = [ys * zs, xs**2, sp.sin(xs * ys)]
fA = [f_sym * a for a in A]
grad_f = [sp.diff(f_sym, v) for v in (xs, ys, zs)]
lhs_div = div(fA); rhs_div = f_sym * div(A) + sum(a * g for a, g in zip(A, grad_f))
lhs_curl = curl(fA); cross = [grad_f[1] * A[2] - grad_f[2] * A[1], grad_f[2] * A[0] - grad_f[0] * A[2], grad_f[0] * A[1] - grad_f[1] * A[0]]
rhs_curl = [f_sym * c + d for c, d in zip(curl(A), cross)]
print("div(fA) - [f div A + A . grad f] =", sp.simplify(lhs_div - rhs_div))
print("curl(fA) - [f curl A + grad f x A] =", [sp.simplify(l - r) for l, r in zip(lhs_curl, rhs_curl)])
assert sp.simplify(lhs_div - rhs_div) == 0 and all(sp.simplify(l - r) == 0 for l, r in zip(lhs_curl, rhs_curl))

## 5. Explore and rebuild

Explore. The field $\mathbf{F} = (-y, x)/(x^2 + y^2)$ is the rotation field divided by $r^2$. Measure its curl with shrinking loops at several points away from the origin. It is zero everywhere you can measure it. Then compute its circulation around the unit circle: $2\pi$. A field with zero curl everywhere should have zero circulation around every loop. Find the loophole (it has to do with where the field is defined), and keep the example in mind: it is the magnetic field of a straight wire, and the puzzle is resolved in notebook 00d and chapter 12.

Rebuild. Close this notebook. On paper, derive the divergence from the flux out of a small box and the $z$ curl from the circulation around a small loop, as in sections 3.3 and 3.4. Then write the two tables of worked examples 4 and 6 from scratch. In a blank editor, write `flux_out_of_square` and `circulation_around_square` and reproduce both tables numerically.